# 04 — RAG Pipeline
## Retrieval Augmented Generation with HyDE + Gemini
We will:
- Load FAISS index from disk
- Connect to Gemini AI
- Build basic retrieval
- Add HyDE for better search
- Add re-ranking for accuracy
- Generate legal answers with sources

In [1]:
import json
import numpy as np
import faiss
from sentence_transformers import SentenceTransformer
from google import genai
from google.genai import types
import os
from dotenv import load_dotenv

# Load environment variables
load_dotenv('../.env')

# Load FAISS index
print("⏳ Loading FAISS index...")
index = faiss.read_index('../vector_store/legal_index.faiss')
print(f"✅ FAISS index loaded!")
print(f"📊 Total vectors : {index.ntotal}")
print()

# Load metadata
print("⏳ Loading metadata...")
with open('../vector_store/metadata.json', 'r', encoding='utf-8') as f:
    chunks = json.load(f)
print(f"✅ Metadata loaded!")
print(f"📊 Total chunks  : {len(chunks)}")
print()

# Load embedding model
print("⏳ Loading embedding model...")
model = SentenceTransformer('all-MiniLM-L6-v2')
print(f"✅ Embedding model loaded!")

C:\Users\mrige\AppData\Local\Packages\PythonSoftwareFoundation.Python.3.12_qbz5n2kfra8p0\LocalCache\local-packages\Python312\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


⏳ Loading FAISS index...
✅ FAISS index loaded!
📊 Total vectors : 56617

⏳ Loading metadata...
✅ Metadata loaded!
📊 Total chunks  : 56617

⏳ Loading embedding model...


Loading weights: 100%|██████████| 103/103 [00:00<00:00, 749.58it/s]


✅ Embedding model loaded!


## Step 1 — Connect to Gemini AI
Setup Google Gemini as our answer generation engine
## Used  FREE Google API key! FROM BELOW WEBSITE 
Website= "aistudio.google.com/api-keys"

##
model="gemini-2.0-flash-lite"    # try first
model="gemini-2.0-flash"         # try second  
model="gemini-1.5-flash"         # try third
model="gemini-1.5-pro"           # try fourth

## Below model woks
model="gemini-2.5-flash"    # try first


In [2]:
import os
import json
import numpy as np
import faiss
from sentence_transformers import SentenceTransformer
from google import genai
from dotenv import load_dotenv

# Load API key from .env file
load_dotenv('../.env', override=True)
GOOGLE_API_KEY = os.getenv('GOOGLE_API_KEY')

# Check key loaded
print(f"✅ Key loaded: {GOOGLE_API_KEY[:8]}...{GOOGLE_API_KEY[-4:]}")

# Connect to Gemini
client = genai.Client(api_key=GOOGLE_API_KEY)

# Test connection
print("⏳ Testing Gemini...")
response = client.models.generate_content(
    model="gemini-2.5-flash",
    contents="Say exactly: Gemini connected successfully!"
)

print("✅ Gemini connected!")
print(f"🤖 {response.text}")

✅ Key loaded: AIzaSyBr...w-AM
⏳ Testing Gemini...
✅ Gemini connected!
🤖 Gemini connected successfully!


In [ ]:
### Important Note: If you encounter an error related to the API key, please ensure that:

# See all available models for your API key
for model in client.models.list():
    print(model.name)

## first output model worked ----  gemini-2.5-flash --- model="gemini-2.5-flash",

models/gemini-2.5-flash
models/gemini-2.5-pro
models/gemini-2.0-flash
models/gemini-2.0-flash-001
models/gemini-2.0-flash-lite-001
models/gemini-2.0-flash-lite
models/gemini-2.5-flash-preview-tts
models/gemini-2.5-pro-preview-tts
models/gemma-3-1b-it
models/gemma-3-4b-it
models/gemma-3-12b-it
models/gemma-3-27b-it
models/gemma-3n-e4b-it
models/gemma-3n-e2b-it
models/gemma-4-26b-a4b-it
models/gemma-4-31b-it
models/gemini-flash-latest
models/gemini-flash-lite-latest
models/gemini-pro-latest
models/gemini-2.5-flash-lite
models/gemini-2.5-flash-image
models/gemini-3-pro-preview
models/gemini-3-flash-preview
models/gemini-3.1-pro-preview
models/gemini-3.1-pro-preview-customtools
models/gemini-3.1-flash-lite-preview
models/gemini-3-pro-image-preview
models/nano-banana-pro-preview
models/gemini-3.1-flash-image-preview
models/lyria-3-clip-preview
models/lyria-3-pro-preview
models/gemini-3.1-flash-tts-preview
models/gemini-robotics-er-1.5-preview
models/gemini-robotics-er-1.6-preview
models/gem

## Step 2 — Basic Retrieval Function
Given a question, find top 5 relevant law sections from FAISS

In [3]:
# Basic retrieval function
def retrieve(query, k=5):
    # Convert question to vector
    query_vector = model.encode([query]).astype('float32')
    
    # Search FAISS
    distances, indices = index.search(query_vector, k=k)
    
    # Return results with metadata
    results = []
    for i, idx in enumerate(indices[0]):
        chunk = chunks[idx]
        results.append({
            'rank'    : i + 1,
            'distance': round(float(distances[0][i]), 4),
            'act'     : chunk['act_title'],
            'section' : chunk['section_id'],
            'heading' : chunk['section_heading'],
            'text'    : chunk['text']
        })
    return results

# Test basic retrieval
query = "punishment for murder"
print(f"🔍 Query: '{query}'")
print("=" * 60)

results = retrieve(query, k=5)
for r in results:
    print(f"\n📌 Rank {r['rank']} (Distance: {r['distance']})")
    print(f"   Act     : {r['act']}")
    print(f"   Section : {r['section']}")
    print(f"   Heading : {r['heading']}")
    print(f"   Text    : {r['text'][:150]}...")
    print("-" * 60)

🔍 Query: 'punishment for murder'

📌 Rank 1 (Distance: 0.7595)
   Act     : THE SASHASTRA SEEMA BAL ACT, 2007
   Section : Section 133.
   Heading : Execution of sentence of death.
   Text    : In executing a sentence of death, a Force Court shall, in its discretion direct that the offender shall suffer death by being hanged by the neck until...
------------------------------------------------------------

📌 Rank 2 (Distance: 0.7955)
   Act     : THE AIRPORTS ECONOMIC REGULATORY AUTHORITY OF INDIA ACT, 2008
   Section : Section 41.
   Heading : Offences by Government Departments.
   Text    :  against and punished accordingly....
------------------------------------------------------------

📌 Rank 3 (Distance: 0.802)
   Act     : THE INDO-TIBETAN BORDER POLICE FORCE ACT, 1992
   Section : Section 133.
   Heading : Form of sentence of death.
   Text    : In awarding a sentence of death, a Force Court shall, in its discretion direct that the offender shall suffer death by being hanged by 

In [4]:
# Reload everything fresh to fix variable clash
import json
import numpy as np
import faiss
from sentence_transformers import SentenceTransformer

# Load FAISS index
index = faiss.read_index('../vector_store/legal_index.faiss')

# Load metadata
with open('../vector_store/metadata.json', 'r', encoding='utf-8') as f:
    chunks = json.load(f)

# Load embedding model with DIFFERENT variable name
embedding_model = SentenceTransformer('all-MiniLM-L6-v2')

print(f"✅ FAISS loaded: {index.ntotal} vectors")
print(f"✅ Chunks loaded: {len(chunks)}")
print(f"✅ Embedding model loaded!")

Loading weights: 100%|██████████| 103/103 [00:00<00:00, 14686.84it/s]


✅ FAISS loaded: 56617 vectors
✅ Chunks loaded: 56617
✅ Embedding model loaded!


In [5]:
# Basic retrieval function using embedding_model
def retrieve(query, k=5):
    # Use embedding_model NOT model
    query_vector = embedding_model.encode([query]).astype('float32')
    
    distances, indices = index.search(query_vector, k=k)
    
    results = []
    for i, idx in enumerate(indices[0]):
        chunk = chunks[idx]
        results.append({
            'rank'    : i + 1,
            'distance': round(float(distances[0][i]), 4),
            'act'     : chunk['act_title'],
            'section' : chunk['section_id'],
            'heading' : chunk['section_heading'],
            'text'    : chunk['text']
        })
    return results

# Test retrieval
query = "punishment for murder"
print(f"🔍 Query: '{query}'")
print("=" * 60)

results = retrieve(query, k=5)
for r in results:
    print(f"\n📌 Rank {r['rank']} (Distance: {r['distance']})")
    print(f"   Act     : {r['act']}")
    print(f"   Section : {r['section']}")
    print(f"   Heading : {r['heading']}")
    print(f"   Text    : {r['text'][:150]}...")
    print("-" * 60)

🔍 Query: 'punishment for murder'

📌 Rank 1 (Distance: 0.7595)
   Act     : THE SASHASTRA SEEMA BAL ACT, 2007
   Section : Section 133.
   Heading : Execution of sentence of death.
   Text    : In executing a sentence of death, a Force Court shall, in its discretion direct that the offender shall suffer death by being hanged by the neck until...
------------------------------------------------------------

📌 Rank 2 (Distance: 0.7955)
   Act     : THE AIRPORTS ECONOMIC REGULATORY AUTHORITY OF INDIA ACT, 2008
   Section : Section 41.
   Heading : Offences by Government Departments.
   Text    :  against and punished accordingly....
------------------------------------------------------------

📌 Rank 3 (Distance: 0.802)
   Act     : THE INDO-TIBETAN BORDER POLICE FORCE ACT, 1992
   Section : Section 133.
   Heading : Form of sentence of death.
   Text    : In awarding a sentence of death, a Force Court shall, in its discretion direct that the offender shall suffer death by being hanged by 

## Step 3 — HyDE (Hypothetical Document Embeddings)
Generate a fake answer first, then search with that fake answer
This improves retrieval accuracy significantly!

# This is the same quota issue — you are hitting the rate limit too fast!
# The free tier allows only 15 requests per minute.

In [8]:
import time

# HyDE without Gemini - Testing logic only
def hyde_retrieve_test(query, k=5):
    print(f"🔍 Original query: '{query}'")
    print()
    
    # Hardcoded fake answer - no Gemini needed for testing!
    fake_answer = """Under Section 302 of the Indian Penal Code, 
    whoever commits murder shall be punished with death or 
    imprisonment for life and shall also be liable to fine. 
    Murder is defined under Section 300 of IPC."""
    
    print(f"📝 Hypothetical answer used for search:")
    print(f"   {fake_answer[:200]}")
    print()
    
    # Search FAISS with fake answer
    print("⏳ Searching FAISS...")
    query_vector = embedding_model.encode([fake_answer]).astype('float32')
    distances, indices = index.search(query_vector, k=k)
    
    results = []
    for i, idx in enumerate(indices[0]):
        chunk = chunks[idx]
        results.append({
            'rank'    : i + 1,
            'distance': round(float(distances[0][i]), 4),
            'act'     : chunk['act_title'],
            'section' : chunk['section_id'],
            'heading' : chunk['section_heading'],
            'text'    : chunk['text']
        })
    return results

# Test it!
query = "punishment for murder"
print("=" * 60)
print("🧪 TESTING HyDE RETRIEVAL")
print("=" * 60)
print()

results = hyde_retrieve_test(query, k=5)

print("📊 HyDE Results:")
print("-" * 60)
for r in results:
    print(f"\n📌 Rank {r['rank']} (Distance: {r['distance']})")
    print(f"   Act     : {r['act']}")
    print(f"   Section : {r['section']}")
    print(f"   Heading : {r['heading']}")
    print(f"   Text    : {r['text'][:150]}...")
    print("-" * 60)

🧪 TESTING HyDE RETRIEVAL

🔍 Original query: 'punishment for murder'

📝 Hypothetical answer used for search:
   Under Section 302 of the Indian Penal Code, 
    whoever commits murder shall be punished with death or 
    imprisonment for life and shall also be liable to fine. 
    Murder is defined under Sectio

⏳ Searching FAISS...
📊 HyDE Results:
------------------------------------------------------------

📌 Rank 1 (Distance: 0.6087)
   Act     : THE INDO-TIBETAN BORDER POLICE FORCE ACT, 1992
   Section : Section 49.
   Heading : Civil offences.
   Text    : Subject to the provisions of section 50, any person subject to this Act who at any place in, or beyond, India commits any civil offence shall be deeme...
------------------------------------------------------------

📌 Rank 2 (Distance: 0.6102)
   Act     : THE SASHASTRA SEEMA BAL ACT, 2007
   Section : Section 49.
   Heading : Civil offences.
   Text    : Subject to the provisions of section 50, any person subject to this Act who

## Step 4 — Full RAG Pipeline
Retrieve relevant law sections + Generate answer with Gemini

## change Model 
## From  model="gemini-2.0-flash-lite",
# TO model="gemini-2.5-flash",

In [ ]:
import time

def ask_legal_question(question):
    print(f"❓ Question: '{question}'")
    print("=" * 60)
    
    # Step 1 - Retrieve relevant sections
    print("⏳ Step 1 - Retrieving relevant law sections...")
    results = retrieve(question, k=5)
    
    # Step 2 - Build context from results
    context = ""
    sources = []
    for r in results:
        context += f"\n\nAct: {r['act']}\n"
        context += f"Section: {r['section']} - {r['heading']}\n"
        context += f"Text: {r['text']}\n"
        sources.append(f"{r['act']} — {r['section']} {r['heading']}")
    
    print(f"✅ Found {len(results)} relevant sections!")
    print()
    
    # Step 3 - Wait to avoid quota limit
    print("⏳ Step 2 - Waiting 5 seconds before Gemini call...")
    time.sleep(5)
    
    # Step 4 - Generate answer with Gemini
    print("⏳ Step 3 - Generating answer with Gemini...")
    prompt = f"""You are an expert Indian legal assistant.
Based on the following law sections, answer the question clearly.
Always cite which Act and Section your answer comes from.

LAW SECTIONS:
{context}

QUESTION: {question}

Provide a clear, accurate answer in simple language."""

    response = client.models.generate_content(
        model="gemini-2.5-flash",        ## changed Model
        contents=prompt
    )
    
    answer = response.text
    
    # Step 5 - Print results
    print()
    print("=" * 60)
    print("⚖️  LEGAL ANSWER:")
    print("=" * 60)
    print(answer)
    print()
    print("📚 Sources used:")
    for s in sources:
        print(f"   → {s}")
    print("=" * 60)
    
    return answer, sources

# TEST the full RAG pipeline!
answer, sources = ask_legal_question(
    "What is the punishment for murder in India?"
)

❓ Question: 'What is the punishment for murder in India?'
⏳ Step 1 - Retrieving relevant law sections...
✅ Found 5 relevant sections!

⏳ Step 2 - Waiting 5 seconds before Gemini call...
⏳ Step 3 - Generating answer with Gemini...

⚖️  LEGAL ANSWER:
Based on the provided law sections, the specific punishment for murder is not explicitly detailed.

However, Section 212 of THE INDIAN PENAL CODE, 1860, when discussing "Harbouring offender," includes "302" (the section pertaining to murder) as an offence and refers to such serious offences as "capital offence" or "punishable with imprisonment for life." This indicates the severe nature of the punishment for murder, though the exact penalty is not stated directly in these sections.

📚 Sources used:
   → THE PROTECTION OF CHILDREN FROM SEXUAL OFFENCES ACT, 2012 — Section 42. Alternative punishment.
   → THE CODE OF CRIMINAL PROCEDURE, 1973 — Section 356. Order for notifying address of previously convicted offender.
   → THE INDIAN PENAL CODE 

In [11]:
# Check which models are available for your account
print("📋 Available Gemini models:")
print()
for m in client.models.list():
    if 'gemini' in m.name.lower():
        print(f"   ✅ {m.name}")

📋 Available Gemini models:

   ✅ models/gemini-2.5-flash
   ✅ models/gemini-2.5-pro
   ✅ models/gemini-2.0-flash
   ✅ models/gemini-2.0-flash-001
   ✅ models/gemini-2.0-flash-lite-001
   ✅ models/gemini-2.0-flash-lite
   ✅ models/gemini-2.5-flash-preview-tts
   ✅ models/gemini-2.5-pro-preview-tts
   ✅ models/gemini-flash-latest
   ✅ models/gemini-flash-lite-latest
   ✅ models/gemini-pro-latest
   ✅ models/gemini-2.5-flash-lite
   ✅ models/gemini-2.5-flash-image
   ✅ models/gemini-3-pro-preview
   ✅ models/gemini-3-flash-preview
   ✅ models/gemini-3.1-pro-preview
   ✅ models/gemini-3.1-pro-preview-customtools
   ✅ models/gemini-3.1-flash-lite-preview
   ✅ models/gemini-3-pro-image-preview
   ✅ models/gemini-3.1-flash-image-preview
   ✅ models/gemini-3.1-flash-tts-preview
   ✅ models/gemini-robotics-er-1.5-preview
   ✅ models/gemini-robotics-er-1.6-preview
   ✅ models/gemini-2.5-computer-use-preview-10-2025
   ✅ models/gemini-embedding-001
   ✅ models/gemini-embedding-2-preview
   ✅ mode

## Step 4 — Full RAG Pipeline (Improved)
### Problems with basic retrieval:
- Plain questions dont match legal terminology in FAISS
- Gemini gets confused with irrelevant sections

### Solution:
- Enrich query with legal terms before searching FAISS
- Give Gemini cleaner focused context
- Result: Much more accurate legal answers!

In [15]:
# Cell 16
import time

# Better retrieval - search specifically for IPC murder sections
def ask_legal_question_v2(question):
    print(f"❓ Question: '{question}'")
    print("=" * 60)
    
    # Step 1 - HyDE without Gemini (use legal terms directly)
    legal_query = f"Indian Penal Code IPC Section 302 murder punishment death life imprisonment {question}"
    
    print("⏳ Step 1 - Retrieving with enhanced query...")
    results = retrieve(legal_query, k=5)
    
    # Step 2 - Build context
    context = ""
    sources = []
    for r in results:
        context += f"\nAct: {r['act']}\n"
        context += f"Section: {r['section']} - {r['heading']}\n"
        context += f"Text: {r['text']}\n"
        sources.append(f"{r['act']} — {r['section']} {r['heading']}")
    
    print(f"✅ Found {len(results)} relevant sections!")
    print()
    
    # Step 3 - Wait before Gemini
    print("⏳ Step 2 - Waiting 10 seconds...")
    time.sleep(10)
    
    # Step 4 - Generate answer
    print("⏳ Step 3 - Generating answer with Gemini...")
    prompt = f"""You are an expert Indian legal assistant.
Based ONLY on these law sections, answer the question.
Be specific. Cite the exact Act and Section.
If answer is not in sections, say so clearly.

LAW SECTIONS:
{context}

QUESTION: {question}

Give a clear direct answer."""

    response = client.models.generate_content(
        model="gemini-2.5-flash",
        contents=prompt
    )
    
    print()
    print("=" * 60)
    print("⚖️  LEGAL ANSWER:")
    print("=" * 60)
    print(response.text)
    print()
    print("📚 Sources:")
    for s in sources:
        print(f"   → {s}")
    print("=" * 60)
    
    return response.text, sources

# Wait 10 seconds then test
print("⏳ Waiting 10 seconds before calling Gemini...")
time.sleep(10)
answer, sources = ask_legal_question_v2(
    "What is the punishment for murder in India?"
)

⏳ Waiting 10 seconds before calling Gemini...
❓ Question: 'What is the punishment for murder in India?'
⏳ Step 1 - Retrieving with enhanced query...
✅ Found 5 relevant sections!

⏳ Step 2 - Waiting 10 seconds...
⏳ Step 3 - Generating answer with Gemini...

⚖️  LEGAL ANSWER:
The punishment for murder is not specified in the provided law sections.

📚 Sources:
   → THE INDIAN PENAL CODE — Section 203. Giving false information respecting an offence committed.
   → THE CODE OF CRIMINAL PROCEDURE, 1973 — Section 356. Order for notifying address of previously convicted offender.
   → THE INDIAN PENAL CODE — Section 212. Harbouring offender.— if a capital offence; if punishable with imprisonment for life, or with imprisonment.
   → THE PROTECTION OF CHILDREN FROM SEXUAL OFFENCES ACT, 2012 — Section 42. Alternative punishment.
   → THE PROBATION OF OFFENDERS ACT, 1958 — Section 3. Power of court to release certain offenders after admonition.


## Step 5 — Retry Logic for Gemini API
### Why do we need retry logic?
- Gemini free tier has rate limits (429 error)
- Gemini servers sometimes busy (503 error)
- Without retry → system crashes on first error
- With retry → system waits and tries again automatically

### How it works:
- Attempt 1 fails → wait 30 seconds → retry
- Attempt 2 fails → wait 60 seconds → retry
- Attempt 3 fails → wait 90 seconds → retry
- All fail → return friendly error message

### Interview tip:
This is called "Exponential Backoff" — a standard
pattern used in ALL production AI systems!

In [14]:
# Cell 17
import time

# Retry logic for Gemini API
def call_gemini_with_retry(prompt, max_retries=3):
    for attempt in range(max_retries):
        try:
            response = client.models.generate_content(
                model="gemini-2.5-flash",
                contents=prompt
            )
            return response.text
        except Exception as e:
            wait_time = (attempt + 1) * 30
            print(f"⚠️ Attempt {attempt + 1} failed: {str(e)[:50]}")
            print(f"⏳ Waiting {wait_time} seconds before retry...")
            time.sleep(wait_time)
    return "❌ All retries failed. Please try again later."

# Test retry logic
print("🧪 Testing Gemini with retry logic...")
response = call_gemini_with_retry("Say: Retry logic working!")
print(f"✅ Response: {response}")

🧪 Testing Gemini with retry logic...
✅ Response: Retry logic working!


## Step 6 — Final Complete RAG Pipeline
Combining everything:
- Enhanced query for better FAISS retrieval
- Gemini answer generation
- Retry logic for reliability
- Sources cited in answer

In [17]:
# Cell 21
import time

# Final complete RAG pipeline
def final_rag(question):
    print(f"❓ Question: '{question}'")
    print("=" * 60)
    
    # Step 1 - Enhanced retrieval
    print("⏳ Step 1 - Retrieving law sections...")
    legal_query = f"Indian law legal section {question}"
    results = retrieve(legal_query, k=5)
    print(f"✅ Found {len(results)} relevant sections!")
    print()
    
    # Step 2 - Build context
    context = ""
    sources = []
    for r in results:
        context += f"\nAct: {r['act']}\n"
        context += f"Section: {r['section']} - {r['heading']}\n"
        context += f"Text: {r['text']}\n"
        sources.append(f"{r['act']} — {r['section']} {r['heading']}")
    
    # Step 3 - Build prompt
    prompt = f"""You are an expert Indian legal assistant.
Answer the question based on these law sections.
Be specific. Cite exact Act and Section numbers.
Use simple language so anyone can understand.

LAW SECTIONS:
{context}

QUESTION: {question}

Give a clear, helpful answer."""

    # Step 4 - Call Gemini with retry
    print("⏳ Step 2 - Generating answer with Gemini...")
    answer = call_gemini_with_retry(prompt)
    
    # Step 5 - Print results
    print()
    print("=" * 60)
    print("⚖️  LEGAL ANSWER:")
    print("=" * 60)
    print(answer)
    print()
    print("📚 Sources:")
    for s in sources:
        print(f"   → {s}")
    print("=" * 60)
    
    return answer, sources

# Test with 3 different legal questions
questions = [
    "What is the punishment for theft in India?",
    "What are the rights of an arrested person?",
    "What is the penalty for income tax evasion?"
]

for q in questions:
    print()
    final_rag(q)
    print()
    print("⏳ Waiting 30 seconds before next question...")
    time.sleep(30)


❓ Question: 'What is the punishment for theft in India?'
⏳ Step 1 - Retrieving law sections...
✅ Found 5 relevant sections!

⏳ Step 2 - Generating answer with Gemini...

⚖️  LEGAL ANSWER:
Based on the law sections you have provided, there is no specific information about the punishment for theft in India.

The sections you've shared from THE SASHASTRA SEEMA BAL ACT, 2007 (Section 49), THE BORDER SECURITY FORCE ACT, 1968 (Section 46), THE INDO-TIBETAN BORDER POLICE FORCE ACT, 1992 (Section 49), and THE NATIONAL SECURITY GUARD ACT, 1986 (Section 45) deal with "civil offences" committed by people who are part of these forces. While theft is a type of civil offence, these sections only state that such people would be tried by a Force Court or Security Force Court if they commit a civil offence, but they don't specify the punishment for theft itself.

The section from THE EPIDEMIC DISEASES ACT, 1897 (Section 3) talks about penalties for disobeying orders related to epidemics, not for theft

## Final Summary — RAG Pipeline Complete

In [18]:
# Cell 23
print("=" * 60)
print("📊 RAG PIPELINE SUMMARY")
print("=" * 60)
print()
print("Components Built:")
print("   ✅ FAISS loaded    : 56,617 vectors")
print("   ✅ Gemini connected: gemini-2.5-flash")
print("   ✅ Basic retrieval : working")
print("   ✅ HyDE retrieval  : working")
print("   ✅ Retry logic     : working")
print("   ✅ Full RAG        : working")
print()
print("Pipeline Flow:")
print("   User Question")
print("   → Enhanced Query")
print("   → FAISS retrieves 5 sections")
print("   → Gemini generates answer")
print("   → Sources cited")
print()
print("Known Limitation:")
print("   ⚠️  FAISS sometimes retrieves military")
print("       law sections for general questions")
print("   ✅ Fix → HyDE with real Gemini call")
print("       improves retrieval accuracy!")
print()
print("=" * 60)
print("✅ Ready to move to 05_multi_agent.ipynb!")
print("=" * 60)

📊 RAG PIPELINE SUMMARY

Components Built:
   ✅ FAISS loaded    : 56,617 vectors
   ✅ Gemini connected: gemini-2.5-flash
   ✅ Basic retrieval : working
   ✅ HyDE retrieval  : working
   ✅ Retry logic     : working
   ✅ Full RAG        : working

Pipeline Flow:
   User Question
   → Enhanced Query
   → FAISS retrieves 5 sections
   → Gemini generates answer
   → Sources cited

Known Limitation:
   ⚠️  FAISS sometimes retrieves military
       law sections for general questions
   ✅ Fix → HyDE with real Gemini call
       improves retrieval accuracy!

✅ Ready to move to 05_multi_agent.ipynb!


## Challenges & Solutions faced in 04_rag_pipeline.ipynb

---

### Challenge 1 — API Key Exposed Publicly
**Problem:**
- Accidentally shared real Gemini API key in public chat
- Someone used the key and exhausted entire free quota
- Got 429 RESOURCE_EXHAUSTED error immediately

**Solution:**
- Deleted compromised key immediately from Google AI Studio
- Created fresh API key
- Stored key ONLY in .env file
- Added .env to .gitignore so it never goes to GitHub

**Lesson Learned:**
- Never share API keys anywhere except .env file
- If key is exposed even for 1 second → delete and rotate immediately
- This is standard security practice in all real companies

---

### Challenge 2 — Wrong Google Package
**Problem:**
- Used old `google.generativeai` package
- Got FutureWarning: "All support has ended"
- Package was deprecated by Google

**Solution:**
- Uninstalled old package: pip uninstall google-generativeai
- Installed new package: pip install google-genai
- Updated all imports to: from google import genai

**Lesson Learned:**
- Always check for deprecated packages
- Google renamed their SDK from google.generativeai → google.genai

---

### Challenge 3 — Quota Exhausted (429 Error)
**Problem:**
- Free tier allows only 15 requests per minute
- Kept retrying quickly after errors
- Got: ClientError 429 RESOURCE_EXHAUSTED
- limit: 0 for all models

**Solution:**
- Added time.sleep(5) before every Gemini call
- Added time.sleep(30) between multiple questions
- Built retry logic with exponential backoff
- Waited 1-2 minutes before retrying after 429 error

**Lesson Learned:**
- Free tier has strict rate limits
- Always add sleep() between API calls
- Exponential backoff is standard production pattern

---

### Challenge 4 — Wrong Gemini Model (404 Error)
**Problem:**
- Used model="gemini-1.5-flash-8b"
- Got: ClientError 404 NOT_FOUND
- "models/gemini-1.5-flash-8b is not found"
- Model not available in India region

**Solution:**
- Listed all available models using client.models.list()
- Found gemini-2.5-flash works for our account
- Updated all model references to gemini-2.5-flash

**Lesson Learned:**
- Not all Gemini models are available in all regions
- Always verify model availability for your region
- gemini-2.5-flash is best available model for India free tier

---

### Challenge 5 — Server Unavailable (503 Error)
**Problem:**
- Got: ServerError 503 UNAVAILABLE
- "This model is currently experiencing high demand"
- System crashed on first failure

**Solution:**
- Built retry logic with 3 attempts
- Added increasing wait times between retries:
  Attempt 1 fails → wait 30 seconds
  Attempt 2 fails → wait 60 seconds
  Attempt 3 fails → wait 90 seconds

**Lesson Learned:**
- Production AI systems always need retry logic
- 503 means server busy, not your fault
- Exponential backoff prevents overwhelming the server

---

### Challenge 6 — Variable Name Clash
**Problem:**
- SentenceTransformer model and Gemini both named "model"
- Got: AttributeError: Model object has no attribute encode
- Python was calling Gemini model instead of embedding model

**Solution:**
- Renamed SentenceTransformer variable to "embedding_model"
- Clear separation between embedding_model and Gemini client
- No more variable name conflicts

**Lesson Learned:**
- Always use descriptive variable names
- embedding_model for SentenceTransformer
- client for Gemini connection

---

### Challenge 7 — Poor Retrieval Quality
**Problem:**
- FAISS returning military/security act sections
- For "punishment for theft" → got Border Security Force Act
- Gemini answered "no specific information found"

**Solution:**
- Enhanced query with legal terminology before FAISS search
- Added "Indian law legal section" prefix to all queries
- Real fix → HyDE with Gemini generates proper legal terms
  before searching FAISS → much better retrieval

**Lesson Learned:**
- Plain natural language queries dont match legal terminology
- Query enrichment significantly improves retrieval accuracy
- HyDE is critical for domain-specific RAG systems like legal AI

---

### Overall Interview Answer:
> "During development we faced 7 major challenges including
> API security breach, deprecated packages, rate limiting,
> regional model availability, server failures, variable conflicts
> and poor retrieval quality. Each challenge taught us something
> important — from API key rotation practices to implementing
> exponential backoff retry logic and query enrichment techniques.
> These real-world problems made the system more robust and
> production-ready."